# 03 · Fine-tuning Whisper on Telugu (Kaggle version)

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/pavanadityav-ai/telugu-asr-whisper/blob/main/notebooks/03_finetune_whisper_kaggle.ipynb)

The same experiment as [notebook 03](03_finetune_whisper.ipynb), adapted for **Kaggle**, which also gives free GPUs.

**Before you run:**
1. Your Kaggle account must be phone-verified (needed for GPU and internet).
2. In the right-hand panel, set **Accelerator → GPU T4 x2** and turn **Internet on**.
3. Click **Save Version → Save & Run All (Commit)**. It runs in the background for about 2 hours, so you can close the tab.

Results are saved in `/kaggle/working`. When it finishes, open the version's **Output** tab and download the files.

In [ ]:
# Setup for Kaggle: one GPU, quiet logs, libraries, project code
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # Kaggle gives two T4s; use one so the settings match Colab
os.environ["HF_HUB_VERBOSITY"] = "error"        # hide "no HF_TOKEN" warnings
os.environ["TRANSFORMERS_VERBOSITY"] = "error"  # hide harmless generation warnings
!pip install -q -U datasets jiwer
if not os.path.isdir("/tmp/telugu-asr-whisper"):
    !git clone -q https://github.com/pavanadityav-ai/telugu-asr-whisper /tmp/telugu-asr-whisper
sys.path.insert(0, "/tmp/telugu-asr-whisper")

OUT = "/kaggle/working"  # files saved here appear in the notebook's Output tab
print("Saving results to:", OUT)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU T4 in the right-hand panel, then run again."
print("GPU:", torch.cuda.get_device_name(0))

## Settings
These are the main knobs. `MAX_STEPS = 1500` with a batch of 16 means the model sees the training set about 10 times.

In [ ]:
BASE_MODEL = "openai/whisper-small"
MAX_STEPS = 1500        # training steps
EVAL_EVERY = 500        # check the validation set every 500 steps
BATCH_SIZE = 16         # clips per step
LEARNING_RATE = 1e-5    # small, because we only adjust an already-trained model
SAVE_DIR = f"{OUT}/whisper-small-te"  # final model is saved in the Output tab

## 1. Load the data

In [ ]:
from src.audio_utils import MAX_SECONDS, SAMPLE_RATE, decode_audio, load_fleurs_telugu
from src.text_utils import normalize_text

ds = load_fleurs_telugu()
ds

## 2. Turn audio and text into model inputs
- **Audio → log-Mel spectrogram:** a picture of which sound frequencies are present over time (80 frequency rows × 3,000 time steps for 30 s). This is what Whisper's encoder reads.
- **Text → token IDs:** the numbers the decoder must learn to output.

Clips longer than 30 s or texts longer than 448 tokens are removed, because Whisper cannot handle them.

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="telugu", task="transcribe")

def prepare(example):
    audio = decode_audio(example["audio"])
    example["input_features"] = processor.feature_extractor(
        audio, sampling_rate=SAMPLE_RATE).input_features[0]
    example["labels"] = processor.tokenizer(normalize_text(example["transcription"])).input_ids
    example["n_seconds"] = len(audio) / SAMPLE_RATE
    return example

def fits(n_seconds, labels):
    return n_seconds <= MAX_SECONDS and len(labels) <= 448

prepared = {}
for split in ["train", "validation"]:
    d = ds[split].map(prepare, remove_columns=ds[split].column_names, desc=split)
    before = len(d)
    d = d.filter(fits, input_columns=["n_seconds", "labels"]).remove_columns("n_seconds")
    print(f"{split}: kept {len(d)} of {before} clips")
    prepared[split] = d

## 3. The data collator
It groups clips into a batch. Labels are padded to the same length, and the padding is set to `-100` so the loss ignores it.

In [ ]:
from dataclasses import dataclass

@dataclass
class SpeechCollator:
    processor: object
    decoder_start_token_id: int

    def __call__(self, features):
        inputs = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(inputs, return_tensors="pt")

        labels = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        label_ids = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        # The model adds the start token itself, so remove it if the tokenizer added it
        if (label_ids[:, 0] == self.decoder_start_token_id).all():
            label_ids = label_ids[:, 1:]
        batch["labels"] = label_ids
        return batch

## 4. Load the model and define the score

In [ ]:
import numpy as np
from transformers import WhisperForConditionalGeneration
from src.metrics import cer, wer

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language = "telugu"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

collator = SpeechCollator(processor, model.config.decoder_start_token_id)
pad_id = processor.tokenizer.pad_token_id

def compute_metrics(pred):
    pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    pred_ids = np.where(pred_ids == -100, pad_id, pred_ids)
    label_ids = np.where(pred.label_ids == -100, pad_id, pred.label_ids)
    P = [normalize_text(t) for t in processor.batch_decode(pred_ids, skip_special_tokens=True)]
    L = [normalize_text(t) for t in processor.batch_decode(label_ids, skip_special_tokens=True)]
    return {"wer": 100 * wer(L, P), "cer": 100 * cer(L, P)}

## 5. Train
- **fp16:** uses 16-bit numbers, which is faster and needs less GPU memory.
- **Gradient checkpointing:** saves memory by recomputing some values instead of storing them.
- **Warm-up:** the learning rate starts near zero and rises over the first 150 steps, so early updates don't damage what Whisper already knows.
- **Best checkpoint:** the model is checked on the validation set every 500 steps, and the version with the **lowest CER** is kept. This protects against overfitting.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir="/tmp/checkpoints",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=150,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    save_steps=EVAL_EVERY,
    save_total_limit=2,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=448,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=prepared["train"],
    eval_dataset=prepared["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
trainer.train()

## 6. Training curve

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

log = trainer.state.log_history
train_log = [(e["step"], e["loss"]) for e in log if "loss" in e]
eval_log = pd.DataFrame([e for e in log if "eval_cer" in e])[["step", "eval_loss", "eval_wer", "eval_cer"]]

plt.figure(figsize=(7, 3))
plt.plot(*zip(*train_log), label="training loss")
plt.plot(eval_log.step, eval_log.eval_loss, "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Fine-tuning whisper-small on Telugu")
plt.tight_layout(); plt.savefig(f"{OUT}/training_curve.png", dpi=150); plt.show()

with open(f"{OUT}/training_log.json", "w") as f:
    json.dump(log, f, indent=1)
eval_log.round(2)

## 7. Save the best model

In [ ]:
trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)

## 8. Final score on the test set
The test set was never used during training or for choosing the checkpoint, so this is a fair score. We use exactly the same code as the baseline notebook.

In [ ]:
del trainer, model
torch.cuda.empty_cache()

from src.asr import load_whisper, transcribe

test = ds["test"]
arrays = [decode_audio(a) for a in test["audio"]]
refs = list(test["transcription"])

MODEL_LABEL = "whisper-small (fine-tuned)"
FILE_TAG = "finetuned_whisper-small"
ft_model, ft_processor = load_whisper(SAVE_DIR)
preds = transcribe(ft_model, ft_processor, arrays, batch_size=16)

In [ ]:
import json
import jiwer
import pandas as pd
from src.metrics import cer, sentence_scores, wer
from src.text_utils import normalize_text, script_profile

R = [normalize_text(r) for r in refs]
H = [normalize_text(p) for p in preds]

scores = {"model": MODEL_LABEL, "wer": wer(R, H), "cer": cer(R, H), "n_clips": len(R)}

# Check our from-scratch metrics against the jiwer library
pairs = [(r, h) for r, h in zip(R, H) if r]
jr, jh = [p[0] for p in pairs], [p[1] for p in pairs]
assert abs(scores["wer"] - jiwer.wer(jr, jh)) < 1e-6, "WER does not match jiwer"
assert abs(scores["cer"] - jiwer.cer(jr, jh)) < 1e-6, "CER does not match jiwer"

print(f"{MODEL_LABEL}:  WER = {100 * scores['wer']:.1f}%   CER = {100 * scores['cer']:.1f}%")

names = getattr(test.features["gender"], "names", None)
rows = []
for clip_id, n, g, r, h in zip(test["id"], test["num_samples"], test["gender"], R, H):
    s = sentence_scores(r, h)
    rows.append({"id": clip_id, "seconds": round(n / 16_000, 2),
                 "gender": names[g] if names else g, "reference": r,
                 "prediction": h, "wer": s["wer"], "cer": s["cer"],
                 "telugu_share": script_profile(h)["telugu"]})
df = pd.DataFrame(rows)
df.to_csv(f"{OUT}/{FILE_TAG}_test_predictions.csv", index=False)
with open(f"{OUT}/{FILE_TAG}_scores.json", "w") as f:
    json.dump(scores, f, indent=2)
print("Saved", f"{OUT}/{FILE_TAG}_test_predictions.csv")

In [ ]:
# Best, middle and worst sentence (by character error rate)
ranked = df.sort_values("cer").reset_index(drop=True)
for label, i in [("BEST", 0), ("MIDDLE", len(ranked) // 2), ("WORST", len(ranked) - 1)]:
    row = ranked.iloc[i]
    print(f"--- {label}  (CER {100 * row.cer:.0f}%)")
    print("REF :", row.reference)
    print("PRED:", row.prediction, "\n")

## Optional: share the model on Hugging Face
Create a free account at huggingface.co, make an access token with *write* permission, then set `PUSH = True`.

In [ ]:
PUSH = False
if PUSH:
    from huggingface_hub import login
    login()
    ft_model.push_to_hub("whisper-small-telugu")
    ft_processor.push_to_hub("whisper-small-telugu")

## Observations

1. 
2. 
3. 